# Блокнот 8. Ансамбли и случайный лес

Курс «Машинное обучение с нуля», модуль 8. Разделы: 1 — урок 8.1, 2 — урок 8.2, 3 — урок 8.3, 4–5 — урок 8.4. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Первая ячейка ставит библиотеки Optuna и SHAP, если их нет.

In [ ]:
%pip install -q optuna shap

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits, load_wine
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(0)

## 1. Мудрость толпы и бутстрэп (урок 8.1)

**Задание 1.** Вероятность, что право большинство из `n` независимых моделей (каждая права с вероятностью `p`): сумма $C_n^k p^k (1-p)^{n-k}$ по `k` от `n // 2 + 1` до `n`.

In [ ]:
def majority_accuracy(n, p):
    return ...  # ваш код

In [ ]:
assert abs(majority_accuracy(3, 0.7) - 0.784) < 1e-9
votes = rng.random((100_000, 5)) < 0.7          # 100 000 советов из 5 магистров
simulated = (votes.sum(axis=1) >= 3).mean()
print(f"Формула: {majority_accuracy(5, 0.7):.4f}, симуляция: {simulated:.4f}")

Доля объектов, не попавших в бутстрэп-выборку, — около $1/e \approx 0{,}368$:

In [ ]:
n = 1000
oob_share = [1 - len(np.unique(rng.integers(0, n, n))) / n for _ in range(200)]
print(f"Вне выборки в среднем: {np.mean(oob_share):.3f}; 1/e = {1 / math.e:.3f}")

## 2. Лес и OOB-оценка (урок 8.2)

In [ ]:
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
models = {"дерево": DecisionTreeClassifier(random_state=0),
          "бэггинг": BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, n_jobs=-1, random_state=0),
          "лес": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)}
for name, m in models.items():
    print(f"{name:8s} тест: {m.fit(X_train, y_train).score(X_test, y_test):.3f}")

In [ ]:
ks = [10, 25, 50, 100, 200, 400]
oob = [RandomForestClassifier(n_estimators=k, oob_score=True, n_jobs=-1, random_state=0)
       .fit(X_train, y_train).oob_score_ for k in ks]
plt.plot(ks, oob, "o-")
plt.xscale("log")
plt.xlabel("число деревьев")
plt.ylabel("OOB accuracy")
plt.show()

## 3. Важность признаков (урок 8.3)

Добавим к винам два столбца шума и сравним важность по примесям (MDI) с перестановочной.

In [ ]:
wine = load_wine()
Xw = np.c_[wine.data, rng.normal(size=len(wine.data)), rng.integers(0, 2, len(wine.data))]
names = list(wine.feature_names) + ["СЛУЧАЙНЫЙ числовой", "СЛУЧАЙНЫЙ 0/1"]
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, wine.target, test_size=0.25, stratify=wine.target, random_state=0)
forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xw_tr, yw_tr)

**Задание 2.** Посчитайте перестановочную важность на тесте (`permutation_importance`, 30 повторов) и выведите таблицу «признак — MDI — перестановочная».

In [ ]:
perm = ...  # ваш код: массив средних падений accuracy
for i in np.argsort(forest.feature_importances_)[::-1]:
    print(f"{names[i]:30s} MDI {forest.feature_importances_[i]:.3f}   перестановка {perm[i]:.3f}")
assert perm[names.index("СЛУЧАЙНЫЙ числовой")] < 0.01

SHAP: как признаки сдвинули прогноз для первого вина из теста.

In [ ]:
import shap

explainer = shap.TreeExplainer(forest)
explanation = explainer(Xw_te[:1])
cls = int(forest.predict(Xw_te[:1])[0])
contrib = explanation.values[0, :, cls]
base = explanation.base_values[0, cls]
print(f"Сорт {cls}: среднее {base:.3f} + вклады {contrib.sum():.3f} = {base + contrib.sum():.3f}; "
      f"predict_proba = {forest.predict_proba(Xw_te[:1])[0, cls]:.3f}")
for i in np.argsort(np.abs(contrib))[::-1][:5]:
    print(f"{names[i]:30s} {contrib[i]:+.3f}")

## 4. Случайный поиск и Optuna (урок 8.4)

In [ ]:
space = {"n_estimators": [100, 200, 400], "max_features": ["sqrt", "log2", 0.3, 0.5],
         "min_samples_leaf": [1, 2, 4], "max_depth": [None, 10, 20]}
search = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1, random_state=0), space,
                            n_iter=10, cv=3, random_state=0).fit(X_train, y_train)
print("Случайный поиск:", search.best_params_, round(search.best_score_, 4))

**Задание 3.** Допишите целевую функцию Optuna: лес с `n_estimators` от 100 до 300, `max_features` от 0,05 до 0,6 и `min_samples_leaf` от 1 до 4; верните среднюю accuracy по 3 фолдам.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    return ...  # ваш код


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=10)
print("Optuna:", study.best_params, round(study.best_value, 4))
assert study.best_value > 0.95

## 5. Вложенная кросс-валидация (урок 8.4)

In [ ]:
inner = GridSearchCV(DecisionTreeClassifier(random_state=0),
                     {"max_depth": [1, 2, 3, 4, 5, None], "min_samples_leaf": [1, 2, 4, 8]}, cv=5)
inner.fit(wine.data, wine.target)
nested = cross_val_score(inner, wine.data, wine.target, cv=5)
print(f"best_score_ (завышен): {inner.best_score_:.3f}; вложенная оценка: {nested.mean():.3f}")